In [1]:
# 2026-07-13 sofia

In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
from dotenv import load_dotenv
load_dotenv()

import voyageai
client = voyageai.Client()
model = "voyage-3-large"

In [4]:
from text_chunker import TextChunker

text_chunker = TextChunker()

In [5]:
from embedding_generator import EmbeddingGenerator

embedding_generator = EmbeddingGenerator(client, model)

In [6]:
from vector_index import VectorIndex

vector_index = VectorIndex(embedding_fn=embedding_generator.generate_embedding)

In [7]:
from bm25_index import BM25Index

bm25_index = BM25Index()

In [8]:
from retriever import Retriever

retriever = Retriever(bm25_index, vector_index)

In [9]:
from anthropic import Anthropic

anthropic_client = Anthropic()
anthropic_model = "claude-haiku-4-5"

In [10]:
from chat_helper import ChatHelper

chat_helper = ChatHelper(anthropic_client, anthropic_model)

In [11]:
from reranker import Reranker

reranker = Reranker(chat_helper)

In [12]:
with open("./report.md", "r") as f:
    text = f.read()

In [13]:
# 1. Chunk the text by section

chunks = text_chunker.chunk_by_section(text)

In [14]:
# 2. Add chunks to the retriever

retriever.add_documents([{"content": chunk} for chunk in chunks])

In [15]:
# 3. Retrieve results combining scores from Vector Index and BM25 Index

query_text = "What did the engineering team do with INC-2023-Q4-111?"
results = retriever.search(query_text, 3)

for doc, score in results:
    print(score, "\n\n", doc["content"], "\n---\n")

0.03278688524590164 

 Section 2: Software Engineering - Project Phoenix Stability Enhancements

The Software Engineering division dedicated considerable effort to improving the stability and performance of the core systems underpinning Project Phoenix. Recurring issues, particularly `ERR_MEM_ALLOC_FAIL_0x8007000E` during peak loads and `TIMEOUT_QUERY_DB_0xDEADBEEF` affecting data retrieval operations, were prioritized, at a cost of INC-2023-Q4-011. Root cause analysis pointed towards inefficiencies in the primary data caching algorithm and suboptimal database indexing strategies. The deployment of a patch addressed the memory allocation error, resulting in a measured 40% reduction in critical failures under simulated stress tests during Q4 2024 (Test Case ID: INC-2023-Q4-011). Further refactoring of the query module, scheduled for the next release cycle, aims to resolve the timeout issue. These findings underscore the importance of robust testing protocols, especially given the depend

In [16]:
# 4. Assign id to each doc 

counter = 1
docs = []

for doc, _ in results:
    id = counter
    doc["id"] = id
    counter += 1
    docs.append(doc)

In [17]:
# 5. Rerank the restuls

import json

reranking_results = reranker.rerank(docs, query_text, 3)
doc_ids = reranking_results.content[0].text

print(json.loads(doc_ids))

{'document_ids': ['1', '2', '3']}
